# Model Selection Notebook for Dengue Transfer Learning Project

### Dataset
Dengue ML datasets track environmental and temporal factors influencing Aedes mosquito breeding and virus transmission in tropical regions like San Juan and Iquitos.

- #### Temporal Features
    - **city**: Location identifier (e.g., 'sj' for San Juan, 'iq' for Iquitos)—captures city-specific mosquito/dengue patterns.
    - **year, weekofyear, week_start_date**: Time granularity for seasonality; dengue peaks during rainy seasons (weekofyear critical for lagged effects).

- #### Vegetation Indices (NDVI)
    - **ndvi_ne, ndvi_nw, ndvi_se, ndvi_sw**: Normalized Difference Vegetation Index by city quadrant. Higher NDVI indicates lush vegetation providing mosquito shade/breeding sites; key for Aedes habitat detection via satellite.

- #### Precipitation \& Water
    - **precipitation_amt_mm**: Rainfall amount—creates standing water breeding sites.
    - **reanalysis_precip_amt_kg_per_m2, reanalysis_sat_precip_amt_mm**: Reanalysis (modeled) precipitation variants confirming observed rain.
    - **station_precip_mm**: Ground station measurements—most direct rain proxy.

- #### Temperature Metrics
    - **reanalysis_air_temp_k, reanalysis_avg_temp_k, reanalysis_max_air_temp_k, reanalysis_min_air_temp_k**: Reanalysis temps in Kelvin; optimal Aedes range 26-32°C accelerates larval development/virus replication.
    - **station_avg_temp_c, station_max_temp_c, station_min_temp_c**: Station temps in Celsius—ground truth validation.
    - **station_diur_temp_rng_c**: Diurnal range; wider swings stress mosquitoes.
    - **reanalysis_tdtr_k**: Temperature diurnal temperature range (reanalysis).

- #### Humidity \& Moisture
    - **reanalysis_dew_point_temp_k**: Dew point—direct humidity proxy; high values (>20°C) favor mosquito survival.
    - **reanalysis_relative_humidity_percent**: Relative humidity %—critical for egg/larval viability.
    - **reanalysis_specific_humidity_g_per_kg**: Absolute moisture content.


### For a fair fight between LightGBM and our LSTM, both models will have the same starting lineup of features. These include seasonal cues, vegetation signals, missing data hints, and early low-case periods—everything else is just how each model likes to learn over time.

### Notebook sections for the fourth project notebook (Feature Selection/Scaling/Pre-Modeling Splits)
1. Get Data
2. Benchmark model (`LightGBM` 1->1 prediction, WFCV)
3. Model Evaluation
    - ***Test A***
        - 1->1 `LightGBM` on city-wise holdout data
        - 1->1 `LSTM` on city-wise holdout data
        - 1->1 `LSTM` transfer learn on `Iquitos` holdout data
    - ***Test B***
        - 52->1 `LightGBM` on city-wise holdout data
        - 52->1 `LSTM` on city-wise holdout data
        - 52->1 `LSTM` transfer learn on `Iquitos` holdout data
4. Model Tuning  (TBC)

In [1]:
import sys
import os
from pathlib import Path
from typing import List, Tuple, Any, Dict, Optional
# import gc
# import itertools
# import logging
# logging.basicConfig(level=logging.INFO)

# Set one level up as project root|
if os.path.abspath("..") not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))
    
from src.config import ProjectConfig  # project config file parser
from src.utils.eda import value_streaks, top_correlations
from src.utils.visualizations import (compute_correlations_matrix,
                display_distributions, random_color, random_colormap,
                display_timeseries)

import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import r2_score, mean_absolute_error

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

import tensorflow as tf

from src.utils.eda import top_correlations, top_vif
from src.utils.utils import _check_feature_presence, load_file, save_file
from src.utils.transform_utils import invert_scaled_X, invert_scaled_y
# from src.preprocessing.clean import (drop_nan_rows, cap_outliers,
#                                     median_groupwise_impute, pipe_clean)

# from src.preprocessing.engineer.base import (reduce_features,
#                                             add_missingness_features,
#                                             low_value_targets)

# from src.preprocessing.engineer.temporal import (circular_time_features,
#                                                 dynamic_temporal_features)

# from src.preprocessing.engineer.pipeline import pipe_engineer

# from src.preprocessing.select import (remove_features, get_wfcv,
#                                         wfcv_feature_autoselect,
#                                         pipe_select)

# from src.preprocessing.preprocess import (encode_categorical,
#                                             time_aware_group_split,
#                                             robust_scale_data)

from src.preprocessing.pipeline import full_preprocess_pipe
from src.schemas.preprocessing import PreprocessOutput

from src.preprocessing.windowing import (make_windows, make_tf_dataset)

from src.utils.visualizations import display_wfcv_folds
from src.utils.data_utils import split_by_feature

import matplotlib.pyplot as plt
# from matplotlib.ticker import AutoMinorLocator

2026-03-30 20:15:36.775884: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# np.random.seed(42)  # keep active for reproductability with model runs (consider adding to main.py)

In [3]:
cnfg = ProjectConfig.load_configuration()
PATH_TO_RAW_DATA = cnfg.data.dirs["raw"]
PATH_TO_INTERMEDIATE_DATA = cnfg.data.dirs["intermediate"]
PATH_TO_PREPROCESSED_DATA = cnfg.data.dirs["processed"]

FILE_TRAIN_RAW= cnfg.data.files["features_train"]
FILE_LABELS_RAW = cnfg.data.files["labels_train"]

FILE_NAN_CLEAN = cnfg.data.files["nan_mask"]
FILE_TRAIN_CLEAN = cnfg.data.files["features_clean"]
FILE_LABELS_CLEAN = cnfg.data.files["labels_clean"]
FILE_TRAIN_ENG = cnfg.data.files["features_eng"]
FILE_LABELS_ENG = cnfg.data.files["labels_eng"]
FILE_TRAIN_SEL = cnfg.data.files["features_selected"]
FILE_LABELS_SEL = cnfg.data.files["labels_eng"]

FILE_X_TRAIN = cnfg.data.files["X_train"]
FILE_X_VALID = cnfg.data.files["X_valid"]
FILE_Y_TRAIN = cnfg.data.files["y_train"]
FILE_Y_VALID = cnfg.data.files["y_valid"]
FILE_GROUP_MASK = cnfg.data.files["test_mask_groups"]

TARGET = cnfg.preprocess.feature_groups["target"]
CITYGROUP_FEAT = cnfg.preprocess.feature_groups["city"]
WEEK_FEAT = cnfg.preprocess.feature_groups["week"]
DATETIME_FEAT = cnfg.preprocess.feature_groups["datetime"]

## Get Data

- Cleaned data

In [4]:
df_train_clean = load_file(path=PATH_TO_INTERMEDIATE_DATA / FILE_TRAIN_CLEAN, datetime_col=DATETIME_FEAT)
df_labels_clean = load_file(path=PATH_TO_INTERMEDIATE_DATA / FILE_LABELS_CLEAN)

In [5]:
all(df_train_clean.index == df_labels_clean.index)

True

- Selected data (last DataFrame)

In [6]:
df_train_select = load_file(path=PATH_TO_INTERMEDIATE_DATA / FILE_TRAIN_SEL, datetime_col=DATETIME_FEAT)
df_labels_select = load_file(path=PATH_TO_INTERMEDIATE_DATA / FILE_LABELS_SEL)

- Preprocessed data

In [7]:
%%time
full_preprocess = full_preprocess_pipe(overwrite_files=True)
X_train = full_preprocess.scaled_data['X_train']
X_valid = full_preprocess.scaled_data["X_valid"]
y_train = full_preprocess.scaled_data['y_train']
y_valid = full_preprocess.scaled_data['y_valid']
grouped_train, grouped_valid = full_preprocess.grouping_masks

INFO:root:Path file present, overwriting.
INFO:root:Saved (1446, 24) shaped data as dengue_features_clean.parquet
INFO:root:Path file present, overwriting.
INFO:root:Saved (1446, 4) shaped data as dengue_labels_clean.parquet
INFO:root:Path file present, overwriting.
INFO:root:Saved (1446, 11) shaped data as dengue_nan_mask.parquet
INFO:root:Path file present, overwriting.
INFO:root:Saved (1438, 34) shaped data as dengue_features_eng.parquet
INFO:root:Path file present, overwriting.
INFO:root:Saved (1438, 4) shaped data as dengue_labels_eng.parquet
INFO:root:Path file present, overwriting.
INFO:root:Saved (1438, 21) shaped data as dengue_features_select.parquet
INFO:root:Selected 21 features from original + engineered feature set of 34 features.
INFO:root:Path file present, overwriting.
INFO:root:Path file present, overwriting.
INFO:root:Path file present, overwriting.
INFO:root:Saved (1230, 21) shaped data as X_train.npy
INFO:root:Path file present, overwriting.
INFO:root:Saved (208, 2

CPU times: user 7.41 s, sys: 176 ms, total: 7.59 s
Wall time: 2.42 s


In [8]:
%%time
# or just load from disc
X_train = np.load(file=PATH_TO_PREPROCESSED_DATA / FILE_X_TRAIN)
X_valid = np.load(file=PATH_TO_PREPROCESSED_DATA / FILE_X_VALID)
y_train = np.load(file=PATH_TO_PREPROCESSED_DATA / FILE_Y_TRAIN)
y_valid = np.load(file=PATH_TO_PREPROCESSED_DATA / FILE_Y_VALID)
test_mask_groups = np.load(file=PATH_TO_PREPROCESSED_DATA / FILE_GROUP_MASK)

groups = df_labels_select[[CITYGROUP_FEAT]]
grouped_train = groups[~test_mask_groups]
grouped_valid = groups[test_mask_groups]

CPU times: user 4.9 ms, sys: 1.07 ms, total: 5.97 ms
Wall time: 4.75 ms


## Benchmark model
- LightGBM 1->1 prediction, WFCV

## Model Evaluation

### Test A
- [ ] 1->1 `LightGBM` on city-wise holdout data (use `get_wfcv` for baseline)
- [ ] 1->1 `LSTM` on city-wise holdout data
- [ ] 1->1 `LSTM` transfer learn on `Iquitos` holdout data
- [ ] 1->1 `LSTM` train with `Iquitos` for transfer learn performance comparison
- [ ] (optional) experiment with `shuffle=False` in `make_tf_windows()` for 1->1 sequences
- [ ] (optional) Multi-horizon (2 and/or 4 weeks) forecasting
    - [ ] Direct 4 target (ie y_t+1 - y_t+4 for `LightGBM`
    - [ ] Direct 4 target (ie `Dense(4)` as last layer)

#### LSTM preprocessing
- [X] convert data to tensors
- [ ] city aware split option for transfer learn

In [9]:
# get tensors
# def to_lstm_tensors(X, y):

# get city aware masks for LSTM splitting

In [10]:
# def split_by_feature(X: np.ndarray,
#                     y: np.ndarray,
#                     group_source: pd.DataFrame,
#                     feature:str) -> tuple[dict, dict]:
#     """
#     A helper to split NumPy feature and target arrays into groups based on a DataFrame column.

#     :param X: Input feature array (e.g., X_train or X_valid).
#     :param y: Input target array aligned with X.
#     :param group_source: DataFrame containing grouping feature aligned with X/y.
#     :param feature: Column name used to split data (e.g., 'city').
#     :return: Tuple of two dictionaries:
#              - X_grouped: {group_value: X_subset}
#              - y_grouped: {group_value: y_subset}
#     """
#     X_grouped = dict()
#     y_grouped = dict()
#     for city in group_source[feature].unique():
#         mask = (group_source[feature] == city).values
#         X_grouped[city] = X[mask]
#         y_grouped[city] = y[mask]
#     return X_grouped, y_grouped

In [11]:
output = split_by_feature(
    X=X_train,
    y=y_train,
    group_source=grouped_train,
    feature=CITYGROUP_FEAT
)

for data in output:
    for k,v in data.items():
        print(f"{k}: {v.shape}")

sj: (822, 21)
iq: (408, 21)
sj: (822,)
iq: (408,)


In [12]:
# def make_tf_dataset(X: np.ndarray,
#                  y: np.ndarray,
#                  batch_size: int | None = None,
#                  shuffle: bool = True
#                 ) -> tf.data.Dataset:
#     """
#     Convert NumPy arrays into a batched `tf.data.Dataset` suitable for LSTM training.
    
#     Handles both windowed sequences (from `make_windows`) and pointwise (1→1) data.
#     Adds a dummy timestep dimension for 2D inputs if needed.
    
#     :param X: Input features array. Shape can be:
#               - `(n_windows, window_size, n_features)` for windowed sequences
#               - `(n_samples, n_features)` for tabular 2D data
#     :param y: Target array. Shape can be:
#               - `(n_windows, horizon)` for windowed sequences
#               - `(n_samples,)` or `(n_samples, 1)` for tabular 2D data
#     :param batch_size: Number of samples per batch. Defaults to config value (e.g., 32).
#     :param shuffle: Whether to shuffle samples/windows between epochs. Defaults to True.

#     :return: A `tf.data.Dataset` yielding tuples `(batch_X, batch_y)`:
#              - `batch_X`: `(batch_size, window_size, n_features)` (or `(batch_size, 1, n_features)` for 1→1)
#              - `batch_y`: `(batch_size, horizon)` (or `(batch_size, 1)` for 1→1)

#     Notes:
#         - Does **not** use `drop_remainder=True` by default; enable if exact batch sizes are required for LSTM.
#         - Prepares data efficiently for GPU/CPU via `.prefetch(tf.data.AUTOTUNE)`.
#         - Compatible with both windowed sequence datasets and non-windowed 1→1 datasets.
#     """
#     batch_size = batch_size or cnfg.preprocess.windowing["batch_size"]

#     if X.ndim == 2:
#         X = X.copy()[:, np.newaxis, :]
#     if y.ndim == 1:
#         y = y.copy()[:, np.newaxis]

#     tf_dataset = tf.data.Dataset.from_tensor_slices(tensors=(X, y))

#     if shuffle:
#         tf_dataset = tf_dataset.shuffle(buffer_size=len(X))
            
#     tf_dataset = tf_dataset.batch(batch_size=batch_size)

#     return tf_dataset.prefetch(tf.data.AUTOTUNE)

In [13]:
X_train.shape, y_train.shape

((1230, 21), (1230,))

In [14]:
X_train_w, y_train_w = make_windows(X=X_train, y=y_train)
tf_data = make_tf_dataset(X=X_train_w, y=y_train_w)  # windowed dataset

In [15]:
# tf_data = make_tf_dataset(X=X_train_sc, y=y_train_sc, shuffle=False)  # regular 1->1 dataset

In [16]:
for batch in tf_data.take(2):
    inputs, targets = batch
    print(f"inputs.shape = {inputs.shape}; targets.shape = {targets.shape}")
    print(f"First window, first step:\n: {inputs[0][0]}...")
    print(f"Targets: {targets[0]}")
    print("=" * 80)
    

inputs.shape = (32, 21, 52); targets.shape = (32, 1)
First window, first step:
: [-0.60609025 -0.60609025 -0.60609025 -0.60609025 -0.28926664 -0.60609025
 -0.60609025 -0.60609025 -0.51754922 -0.50108027 -0.60609025 -0.26431862
 -0.45917411  1.34442134  2.71443969  2.31087196  1.12298724 -0.27736334
 -0.17300558 -0.57282622  0.06538665 -0.18393054 -0.00962048  1.05124129
  0.38384085 -0.28339652 -0.54005136  0.6186458  -0.0218499  -0.27622192
  0.0508744   1.37344584 -0.00733765  0.50564592  0.1379479   0.30165913
  0.28649464 -0.11202152 -0.60609025 -0.20072561  0.86943052 -0.27654804
 -0.60609025 -0.60609025 -0.60609025 -0.5025478  -0.36150177  0.30182218
  0.38318862 -0.60609025 -0.60609025 -0.12832742]...
Targets: [0.12063591]
inputs.shape = (32, 21, 52); targets.shape = (32, 1)
First window, first step:
: [-0.2794831   0.75806123  2.71443969 -0.57168481  0.42884513 -0.60609025
 -0.60609025  0.05185276  0.72659084  0.13077331 -0.60609025 -0.60609025
 -0.60609025 -0.21165056 -0.60609

2026-03-30 20:15:41.348588: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Test B
- [ ] 52->1 `LightGBM` on city-wise holdout data
- [ ] 52->1 `LSTM` on city-wise holdout data
- [ ] 52->1 `LSTM` transfer learn on `Iquitos` holdout data:
    - [ ] experiment with `shuffle=False` in `make_tf_windows()` for `Iquitos`
    - [ ] Compare convergence speed and final performance for global LSTM vs transfer LSTM:
        - [ ] if transfer converges faster but ends similar → better training dynamics on split data
        - [ ] if transfer heavily outperforms → likely inductive bias (ie shared data structure)
- [ ] 52->1 `LSTM` train with `Iquitos` for transfer learn performance comparison
- [ ] (optional) Multi-horizon (2 and/or 4 weeks) forecasting
    - [ ] Direct 4 target (ie y_t+1 - y_t+4 for `LightGBM`
    - [ ] Direct 4 target (ie `Dense(4)` as last layer)

#### Window splits for 52->1 forecasting
- [X] pandas utility for `LightGBM`
- [X] keras tensor windows for `LSTM`

In [17]:
# def make_windows(X: np.ndarray,
#                  y: np.ndarray,
#                  window_size: int | None = None,
#                 stride: int | None = None,
#                 horizon: int | None = None) -> tuple[np.ndarray, np.ndarray]:
#     """
#     Create zero-copy sliding windows for time series forecasting using stride tricks.
    
#     Past `window_size` timesteps predict the next week's `horizon` timesteps ahead.
    
#     :param X: Input features array of shape `(n_timesteps, n_features)`.
#     :param y: Target array of shape `(n_timesteps,)` or `(n_timesteps, n_targets)`.
#     :param window_size: Number of past timesteps for input window. Falls back to 
#                         config file settings (default: 52).
#     :param stride: Step size between consecutive windows. Falls back to 
#                    config file settings (default: 1).
#     :param horizon: Number of future timesteps to predict. Falls back to 
#                     config file settings  (default: 1).
    
#     :return: Tuple of zero-copy window arrays:
#              - ``X_windows``: Shape `(n_windows, window_size, n_features)`
#              - ``y_windows``: Shape `(n_windows, horizon)`
#     """
#     settings = cnfg.preprocess.windowing
#     window_size = window_size or settings["input_weeks"]
#     stride = stride or settings["stride"]
#     horizon = horizon or settings["output_weeks"]
    
#     strider = np.lib.stride_tricks.sliding_window_view
#     X_windows = strider(X, window_shape=window_size, axis=0)[::stride]
    
#     y_start = window_size 
#     y_windows = strider(y[y_start:], window_shape=horizon, axis=0)[::stride]

#     cuttof = min(X_windows.shape[0], y_windows.shape[0])
    
#     # APPROACH BEFORE lib.stride_tricks.sliding_window_view (more memory efficient)
#     # n_windows = int(np.ceil((len(X) - window_size - horizon + 1) / stride))
#     # X_windows = np.empty((n_windows, window_size, X.shape[1]))
#     # y_windows = np.empty((n_windows, horizon))

#     # for w in range(n_windows):
#     #     start = stride * w
#     #     X_windows[w] = X[start:start+window_size]
#     #     y_windows[w] = y[start+window_size:start+window_size+horizon]

#     return X_windows[:cuttof], y_windows[:cuttof]

In [18]:
%%time
X_train_w, y_train_w = make_windows(X=X_train, y=y_train)

CPU times: user 176 μs, sys: 10 μs, total: 186 μs
Wall time: 193 μs


In [19]:
X_train_w.shape, y_train_w.shape

((1178, 21, 52), (1178, 1))

In [20]:
X_train_w_4, y_train_w_4 = make_windows(X=X_train, y=y_train, horizon=4)
X_train_w_4.shape, y_train_w_4.shape

((1175, 21, 52), (1175, 4))

# TODO:
- Target processing
- [X] Process zero/low value target value streaks
- [X] RobustScaler (Targets)
- [X] Predict log1p(total_cases) to reduce zero target influence (log transform targets)
- [ ] (Optional) LSTM specific - if LSTM results are sub-optimal, experiment with downweighting affected period data (ie 0.3 or more)

# TODO Research before applying:
- **Final Preprocessing tactics for outliers:**
    - Target ("total_cases")
        - If tree models used (eg LightGBM) - no issue, trees are not sensitive to outliers:
            - [ ] use huber loss for extra safety when handling tails
            - [X] `city` one-hot encoding (no string feature remains)
        - RNNs (eg LSTM) are outlier sensitive (gradient instability, hidden state patterns loose importance at peaks, scaling):
            - [X] Log transform
            - [X] Scale (RobustScaler  with IQR is more outlier resistant)
            - [ ] apply huber loss
            - [X] groupby `city` or  one-hot encoding if passing entire dataset (no string feature remains)
            - [X]  windowed input with:
                -  [ ] 52 ime steps (if single step/week forecast)